[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/raya-lucaria/ia_o26/blob/main/course/10_deep_learning/_assets/04_finetuning_llm.ipynb)

# Notebook 4 · Fine-tuning de un LLM con LoRA: tu dataset

- Unidad 10 de *deep learning*, clase 3. Un **LLM chico** (Qwen2.5, 0.5 mil millones de parámetros) aprende **tu** dataset de texto con LoRA.
- Pregunta: **¿qué aprende un modelo de lenguaje de unos cientos de ejemplos, y cómo lo sabes sin engañarte?**
- Es el gemelo del notebook 3: las mismas secciones, pero con texto en vez de imágenes.

**Tres pasos**

1. En Colab: «Entorno de ejecución → Cambiar tipo de entorno → **T4 GPU** → Guardar».
2. Arrastra tu `texto.zip` (el de la tarea) al panel 📁 **Archivos** de la izquierda. Si no tienes zip, no pasa nada: el notebook arma uno de demostración (texto → SQL).
3. «Entorno de ejecución → **Ejecutar todo**».

| `MODO` | Modelo | Qué corre | Tiempo en una T4 |
|---|---|---|---|
| `"corto"` (por omisión) | Qwen2.5-0.5B-Instruct | un LoRA en atención, lo esencial | ≈ 20–25 min |
| `"largo"` | Qwen2.5-1.5B-Instruct | más pasos, más datos, LoRA también en el MLP | ≈ 45–60 min |

**Test, una sola vez.** Todo se decide con *validation*. El *test* se usa **solo al final (§08), una vez**, con el modelo ya entrenado.

**Mapa**

| § | Sección | Qué te llevas |
|---|---|---|
| 01 | Instalar y configurar | `MODO`, GPU, modelo |
| 02 | Tu zip | revisar el formato, ver ejemplos, partir 80/10/10 |
| 03 | Antes | qué contesta el modelo base |
| 04 | LoRA por dentro | las matemáticas y cuántos parámetros entrenas de verdad |
| 05 | Entrenar | la loss solo en el *completion*, curvas train / validation |
| 06 | Ver LoRA | rango de $\Delta W$, qué capas cambiaron, la perilla $\lambda$, *merge* |
| 07 | Métricas | qué mide cada número y cuál es su trampa |
| 08 | Testeo | test una vez, olvido, descargar tu adaptador, qué mover si salió mal |

## 01 · Instalar y configurar

- **Elige el modo en la primera línea de código.** `MODO = "corto"` tarda ≈ 20–25 min en una T4; `MODO = "largo"` usa un modelo tres veces más grande y entrena más (≈ 45–60 min).
- La celda instala solo lo que falta. No actualiza lo que Colab ya trae.

| Paquete | Para qué |
|---|---|
| `transformers` | el LLM, su tokenizer y su plantilla de chat |
| `peft` | los adaptadores LoRA |
| `accelerate` | carga eficiente del modelo en la GPU |
| `safetensors` | guardar el adaptador en un archivo seguro |

In [ ]:
MODO = "corto"   # "corto" (≈ 20–25 min en T4) o "largo" (≈ 45–60 min)
LARGO = MODO == "largo"

# Instala solo lo que falta; no actualiza lo que Colab ya trae.
import importlib.util
import subprocess
import sys

FALTAN = [p for p in ["transformers", "peft", "accelerate", "safetensors"]
          if importlib.util.find_spec(p) is None]
if FALTAN:
    print("Instalando:", ", ".join(FALTAN))
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *FALTAN],
                   check=True)
else:
    print("No falta nada que instalar.")
print("MODO =", MODO, "· LARGO =", LARGO)

- **Necesitas GPU.** Si aparece el aviso, elige T4 GPU en «Cambiar tipo de entorno» y ejecuta todo de nuevo.
- `DTYPE` es **fp16** en la GPU: 2 bytes por peso, la mitad que fp32. La T4 no tiene bf16, el formato con el que se entrenó Qwen; fp16 funciona, pero tiene menos rango, y por eso en §05 hay un `GradScaler`.
- `RAPIDO=1` es solo para la prueba local: usa un Qwen **diminuto** con pesos al azar, en CPU. Sus respuestas son basura; sirve para ver que el código corre.

In [ ]:
import os
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import peft
import transformers
import huggingface_hub

# RAPIDO=1: prueba local en CPU con un Qwen diminuto (segundos).
RAPIDO = os.environ.get("RAPIDO") == "1"
HAY_GPU = torch.cuda.is_available()
DEVICE = torch.device("cuda" if HAY_GPU and not RAPIDO else "cpu")
CUDA = DEVICE.type == "cuda"
# fp16 en la GPU (la T4 no tiene bf16); fp32 en CPU.
DTYPE = torch.float16 if CUDA else torch.float32

if not HAY_GPU and not RAPIDO:
    print("=" * 72)
    print("  ¡AVISO! ESTE ENTORNO NO TIENE GPU.")
    print("  Entrenar un LLM en CPU tardaría horas. En Colab:")
    print("    Entorno de ejecución → Cambiar tipo de entorno → T4 GPU")
    print("    → Guardar, y luego Entorno de ejecución → Ejecutar todo.")
    print("=" * 72)

for nombre, mod in [("torch", torch), ("transformers", transformers),
                    ("peft", peft), ("huggingface_hub", huggingface_hub)]:
    print(f"{nombre:16s}{mod.__version__}")
print("DEVICE =", DEVICE, "· DTYPE =", DTYPE, "· RAPIDO =", RAPIDO)
if HAY_GPU:
    props = torch.cuda.get_device_properties(0)
    print(f"GPU: {props.name} · {props.total_memory / 2**30:.1f} GiB")

| Modo | Modelo | Parámetros | Capas con LoRA |
|---|---|---:|---|
| corto | `Qwen/Qwen2.5-0.5B-Instruct` | ≈ 0.5 mil millones | atención: `q_proj`, `k_proj`, `v_proj`, `o_proj` |
| largo | `Qwen/Qwen2.5-1.5B-Instruct` | ≈ 1.5 mil millones | atención + MLP: `gate_proj`, `up_proj`, `down_proj` |

- **Instruct** quiere decir que el modelo ya fue ajustado para conversar: le das un mensaje y contesta. Tú vas a ajustarlo **otra vez**, a tu tarea.
- `P` junta los tamaños del modo:
  - `max_len`: tokens máximos por ejemplo; lo que pase de ahí se corta y **no se aprende**.
  - `batch` × `acum`: ejemplos por paso. Se procesan `batch` a la vez y se acumulan `acum` gradientes antes de actualizar: así cabe en la memoria un batch efectivo de 8.
  - `pasos`, `lr`, `val_cada`: cuántas actualizaciones, con qué learning rate y cada cuántas se mide validation.
  - `n_val_loss`, `n_eval`: cuántos ejemplos de validation (y de test) se usan para la loss y para generar.
  - `max_new`: tope de tokens que el modelo puede generar por respuesta.
  - `rango`, `alfa`, `modulos`: la configuración de LoRA ($r=8$, $\alpha=16$), explicada en §04.
- En `MODO = "largo"` el pico de memoria **sí sube**: el modelo de 1.5B ocupa ≈ 3 GB en fp16, contra ≈ 1 GB del de 0.5B. Cabe en la T4.
- Colores: azul es train, naranja es validation y verde es test.

In [ ]:
# El modelo base. Los dos Qwen2.5-Instruct son Apache-2.0.
if RAPIDO:
    MODELO = "trl-internal-testing/tiny-Qwen2ForCausalLM-2.5"
elif LARGO:
    MODELO = "Qwen/Qwen2.5-1.5B-Instruct"
else:
    MODELO = "Qwen/Qwen2.5-0.5B-Instruct"
DEMO = "b-mc2/sql-create-context"     # el dataset del demo (CC BY 4.0)

# Tamaños por modo. pasos = actualizaciones del adaptador; cada paso ve
# batch × acum ejemplos.
ATENCION = ["q_proj", "k_proj", "v_proj", "o_proj"]
if RAPIDO:
    P = dict(n_demo=200, max_len=128, batch=2, acum=1, pasos=6, lr=2e-4,
             val_cada=3, n_val_loss=8, n_eval=4, max_new=16,
             rango=8, alfa=16, modulos=ATENCION)
elif LARGO:
    P = dict(n_demo=3000, max_len=512, batch=4, acum=2, pasos=800, lr=2e-4,
             val_cada=50, n_val_loss=200, n_eval=150, max_new=192,
             rango=8, alfa=16, modulos=ATENCION)
else:
    P = dict(n_demo=1000, max_len=512, batch=4, acum=2, pasos=300, lr=2e-4,
             val_cada=25, n_val_loss=100, n_eval=100, max_new=128,
             rango=8, alfa=16, modulos=ATENCION)
if LARGO:                               # ➕ LoRA también en el MLP
    P["modulos"] = ATENCION + ["gate_proj", "up_proj", "down_proj"]
display(pd.Series({k: str(v) for k, v in P.items()}, name="valor").to_frame())

# Un color fijo por conjunto, igual que en los notebooks 2 y 3.
COLORES = {"train": "#1f77b4", "validation": "#ff7f0e", "test": "#2ca02c"}


def semilla(s: int):
    """Fija las semillas de random, NumPy y PyTorch (CPU y GPU)."""
    random.seed(s)
    np.random.seed(s)
    torch.manual_seed(s)        # también siembra la GPU


def mostrar(df):
    """Una tabla de texto: celdas alineadas a la izquierda y con saltos."""
    try:
        display(df.style.set_properties(
            **{"text-align": "left", "white-space": "pre-wrap"}))
    except Exception:           # sin jinja2 o fuera de Jupyter
        with pd.option_context("display.max_colwidth", 80):
            display(df)


semilla(0)
plt.style.use("default")
plt.rcParams.update({"axes.grid": True, "grid.alpha": 0.3,
                     "figure.dpi": 80, "savefig.dpi": 80,
                     "axes.spines.top": False, "axes.spines.right": False})
try:                            # en Colab: figuras en JPEG, el notebook pesa menos
    from matplotlib_inline.backend_inline import set_matplotlib_formats
    set_matplotlib_formats("jpeg", pil_kwargs={"quality": 80})
except Exception:
    pass

## 02 · Tu zip

**Dónde subirlo.** El zip es el de la tarea (`texto.zip`: `datos.jsonl`, `prompts.txt`, `LEEME.md` en la raíz).

1. Abre el panel 📁 **Archivos**, a la izquierda de Colab.
2. **Arrastra** tu `texto.zip` adentro y espera a que termine de subir (el círculo de abajo se llena).
3. Ejecuta. La celda lo encuentra sola en `/content`.

Si pesa mucho, súbelo a tu Drive y quita los `#` de las líneas de Drive en la celda. Si tu zip se llama distinto, escribe su ruta en `RUTA_ZIP`.

**Sin zip no pasa nada:** la celda arma `texto_demo.zip` en el mismo formato, con `P["n_demo"]` ejemplos de `b-mc2/sql-create-context` (1,000 en modo corto, 3,000 en largo) (una tabla y una pregunta → la consulta SQL). Es una buena tarea para ver el cambio: las respuestas son cortas y se pueden comparar letra por letra.

**Si subes un zip de imágenes** a este notebook, la celda se detiene y te da el enlace al notebook 3.

In [ ]:
RUTA_ZIP = ""    # vacío: busca texto.zip; si no hay, arma el demo
# Si lo subiste a Drive, quita los # de estas dos líneas:
# from google.colab import drive; drive.mount("/content/drive")
# RUTA_ZIP = "/content/drive/MyDrive/texto.zip"
# O súbelo con un botón (detiene «Ejecutar todo» hasta que elijas):
# from google.colab import files; RUTA_ZIP = list(files.upload())[0]

import glob
import json
import zipfile

NB3 = ("https://colab.research.google.com/github/raya-lucaria/ia_o26/blob/"
       "main/course/10_deep_learning/_assets/03_finetuning_lora.ipynb")


def tipo_de_zip(ruta):
    """'texto', 'imagenes' o None, según lo que traiga en la raíz."""
    try:
        nombres = set(zipfile.ZipFile(ruta).namelist())
    except (zipfile.BadZipFile, OSError):
        return None
    if "datos.jsonl" in nombres:
        return "texto"
    return "imagenes" if "metadata.csv" in nombres else None


def buscar_zip():
    """El zip de texto que subiste: texto.zip, o cualquier .zip con
    datos.jsonl, en /content o en la carpeta actual."""
    carpetas = list(dict.fromkeys(["/content", os.getcwd()]))
    for c in carpetas:
        if os.path.exists(os.path.join(c, "texto.zip")):
            return os.path.join(c, "texto.zip")
    zips = [z for c in carpetas for z in sorted(glob.glob(f"{c}/*.zip"))
            if not z.endswith("texto_demo.zip")]
    for z in zips:
        if tipo_de_zip(z) == "texto":
            return z
    imagenes = [z for z in zips if tipo_de_zip(z) == "imagenes"]
    if imagenes:
        raise ValueError(
            f"{imagenes[0]} es un zip de IMÁGENES (trae metadata.csv).\n"
            f"Este notebook es de texto; las imágenes van en el notebook 3:\n"
            f"{NB3}")
    return None

In [ ]:
def armar_demo(ruta, n):
    """Arma un texto.zip de demostración, en el formato de la tarea, con n
    ejemplos de b-mc2/sql-create-context (pregunta + tablas → SQL)."""
    from huggingface_hub import hf_hub_download
    archivo = hf_hub_download(DEMO, "sql_create_context_v4.json",
                              repo_type="dataset")
    with open(archivo, encoding="utf-8") as f:
        filas = random.Random(0).sample(json.load(f), n)
    datos = [{"prompt": f"Tablas: {e['context']}\nPregunta: {e['question']}",
              "completion": e["answer"]} for e in filas]
    # Cuatro prompts que no están en el dataset; «\n» es un salto de línea.
    prompts = [
        r"Tablas: CREATE TABLE students (name VARCHAR, age INTEGER)\n"
        r"Pregunta: How many students are older than 20?",
        r"Tablas: CREATE TABLE courses (id INTEGER, title VARCHAR, "
        r"credits INTEGER)\nPregunta: What is the title of course 7?",
        r"Tablas: CREATE TABLE songs (title VARCHAR, artist VARCHAR, "
        r"year INTEGER)\nPregunta: Which artists released songs in 1999?",
        r"Tablas: CREATE TABLE cities (name VARCHAR, country VARCHAR, "
        r"population INTEGER)\nPregunta: What is the total population of "
        r"cities in Mexico?"]
    leeme = (
        "# Mi dataset\n"
        "- **Qué quiero que aprenda el modelo:** escribir la consulta SQL "
        "que responde una pregunta sobre unas tablas\n"
        "- **Ruta:** texto\n"
        f"- **Fuente:** https://huggingface.co/datasets/{DEMO}\n"
        "- **Licencia:** CC BY 4.0\n"
        f"- **Cuántos ejemplos y cómo los elegí:** {n} al azar con semilla 0\n")
    with zipfile.ZipFile(ruta, "w", zipfile.ZIP_DEFLATED) as z:
        z.writestr("datos.jsonl", "".join(
            json.dumps(e, ensure_ascii=False) + "\n" for e in datos))
        z.writestr("prompts.txt", "\n".join(prompts) + "\n")
        z.writestr("LEEME.md", leeme)
    return ruta


if RUTA_ZIP:
    assert os.path.exists(RUTA_ZIP), f"no existe {RUTA_ZIP}: revisa la ruta"
    if tipo_de_zip(RUTA_ZIP) == "imagenes":
        raise ValueError(f"{RUTA_ZIP} es un zip de IMÁGENES: úsalo en el "
                         f"notebook 3:\n{NB3}")
else:
    RUTA_ZIP = buscar_zip()
    if RUTA_ZIP is None:
        print("No encontré tu texto.zip: armo el demo (texto → SQL).")
        RUTA_ZIP = armar_demo(os.path.join(os.getcwd(), "texto_demo.zip"),
                              P["n_demo"])
ES_DEMO = RUTA_ZIP.endswith("texto_demo.zip")
print("Uso:", RUTA_ZIP, "(demo)" if ES_DEMO else "(tu zip)")

- Esta es **la misma función** de la página de la tarea, copiada sin cambios. Imprime `OK` o se detiene diciendo qué está mal: arréglalo en tu zip, vuelve a subirlo y ejecuta otra vez.
- Después se leen los tres archivos: la ficha `LEEME.md`, los ejemplos y los prompts de prueba.
- **Saltos de línea en `prompts.txt`:** cada línea es un prompt. Si tu prompt lleva saltos de línea (como el demo: `Tablas: …` y luego `Pregunta: …`), escríbelos como `\n`, con dos caracteres; el notebook los convierte.

**Cómo leer la tabla de largos:** si el máximo de `prompt` + `completion` pasa de ≈ 2,000 caracteres (≈ 500 tokens), esos ejemplos se cortan en §05 y su final no se aprende.

In [ ]:
# La función de la tarea, copiada literal: dice OK o qué está mal.
import csv, io, json, zipfile
from PIL import Image

def revisar_zip(ruta):
    """Dice si el zip sigue el formato de la tarea. No cambia nada."""
    z = zipfile.ZipFile(ruta)
    nombres = set(z.namelist())
    faltan = {"prompts.txt", "LEEME.md"} - nombres
    assert not faltan, f"faltan en la raíz del zip: {sorted(faltan)}"
    prompts = z.read("prompts.txt").decode("utf-8").split("\n")
    prompts = [p for p in prompts if p.strip()]
    assert 3 <= len(prompts) <= 5, f"prompts.txt: {len(prompts)} prompts, van de 3 a 5"
    if "metadata.csv" in nombres:                     # imágenes
        filas = list(csv.DictReader(io.StringIO(z.read("metadata.csv").decode("utf-8"))))
        assert filas and set(filas[0]) == {"file_name", "text"}, "columnas: file_name,text"
        assert len(filas) >= 50, f"solo {len(filas)} imágenes; mínimo 50"
        for fila in filas:
            assert fila["file_name"] in nombres, f"no está {fila['file_name']}"
            assert fila["text"].strip(), f"{fila['file_name']} sin texto"
            Image.open(z.open(fila["file_name"])).verify()
        print(f"OK · imágenes: {len(filas)} · prompts: {len(prompts)}")
    elif "datos.jsonl" in nombres:                    # texto
        lineas = z.read("datos.jsonl").decode("utf-8").splitlines()
        ejemplos = [json.loads(l) for l in lineas if l.strip()]
        assert len(ejemplos) >= 200, f"solo {len(ejemplos)} ejemplos; mínimo 200"
        for i, ej in enumerate(ejemplos, 1):
            assert set(ej) == {"prompt", "completion"}, f"línea {i}: llaves {sorted(ej)}"
            assert ej["completion"].strip(), f"línea {i}: completion vacío"
        print(f"OK · ejemplos: {len(ejemplos)} · prompts: {len(prompts)}")
    else:
        raise AssertionError("falta metadata.csv (imágenes) o datos.jsonl (texto)")

revisar_zip(RUTA_ZIP)

In [ ]:
_z = zipfile.ZipFile(RUTA_ZIP)
EJEMPLOS = [json.loads(l) for l in
            _z.read("datos.jsonl").decode("utf-8").splitlines() if l.strip()]
# En prompts.txt, «\n» escrito con dos caracteres es un salto de línea.
PROMPTS = [p.replace("\\n", "\n").strip() for p in
           _z.read("prompts.txt").decode("utf-8").split("\n") if p.strip()]
LEEME = _z.read("LEEME.md").decode("utf-8")
try:
    from IPython.display import Markdown
    display(Markdown(LEEME))
except ImportError:
    print(LEEME)

largos = pd.DataFrame({
    "prompt": [len(e["prompt"]) for e in EJEMPLOS],
    "completion": [len(e["completion"]) for e in EJEMPLOS]})
print(f"{len(EJEMPLOS)} ejemplos · {len(PROMPTS)} prompts de prueba")
display(largos.describe().loc[["mean", "50%", "max"]].round(0)
        .rename(index={"mean": "promedio (caracteres)",
                       "50%": "mediana", "max": "máximo"}))


def recorta(texto, n=300):
    return texto if len(texto) <= n else texto[:n] + " …"


muestra = random.Random(1).sample(EJEMPLOS, 3)
mostrar(pd.DataFrame([{"prompt": recorta(e["prompt"]),
                       "completion": recorta(e["completion"])}
                      for e in muestra]))

| Parte | Cuánto | Para qué |
|---|---:|---|
| train | 80 % | lo único que ve el entrenamiento |
| validation | 10 % | medir mientras decides: curvas, métricas, la perilla $\lambda$ |
| test | 10 % | **una sola vez**, al final (§08) |

- La semilla 0 fija qué ejemplo cae en cada parte. Si vuelves a correr, los conjuntos son los mismos.
- **¿Por qué el test una sola vez?** Cada vez que miras el test y cambias algo por lo que viste, el test se vuelve parte del entrenamiento y deja de medir lo que el modelo hace con datos nuevos.

In [ ]:
# 80 / 10 / 10 con semilla fija: siempre los mismos ejemplos en cada parte.
orden = np.random.default_rng(0).permutation(len(EJEMPLOS))
n_10 = max(1, round(0.1 * len(EJEMPLOS)))
DATOS = {"validation": [EJEMPLOS[i] for i in orden[:n_10]],
         "test": [EJEMPLOS[i] for i in orden[n_10:2 * n_10]],
         "train": [EJEMPLOS[i] for i in orden[2 * n_10:]]}
display(pd.Series({k: len(v) for k, v in DATOS.items()},
                  name="ejemplos").to_frame())

## 03 · Antes: qué contesta el modelo base

- **Qué vas a ver:** las respuestas del modelo **sin entrenar** a tus prompts de prueba y a tres ejemplos de validation.
- Es la fila de referencia: en §06 y §08 vuelven estos mismos prompts, ya con LoRA.
- La celda descarga el modelo (≈ 1 GB en modo corto, ≈ 3 GB en largo) y lo congela: ningún peso de $W_0$ se va a mover.

In [ ]:
import contextlib
import gc
import math
import time

from packaging.version import Version
from transformers import AutoModelForCausalLM, AutoTokenizer

# transformers ≥ 4.56 llama «dtype» a lo que antes era «torch_dtype».
_DT = ("dtype" if Version(transformers.__version__) >= Version("4.56")
       else "torch_dtype")
TOK = AutoTokenizer.from_pretrained(MODELO)
if TOK.pad_token is None:
    TOK.pad_token = TOK.eos_token
LM = AutoModelForCausalLM.from_pretrained(MODELO, **{_DT: DTYPE}).to(DEVICE)
LM.requires_grad_(False)                 # congelado: solo LoRA va a entrenar
LM.eval()
# Generación greedy: sin azar ni penalizaciones (Qwen trae sampling por omisión).
_g = LM.generation_config
_g.do_sample, _g.temperature, _g.top_p, _g.top_k = False, None, None, None
_g.repetition_penalty = None
n_lm = sum(p.numel() for p in LM.parameters())
print(f"{MODELO}: {n_lm / 1e6:,.0f} M parámetros · "
      f"{n_lm * LM.dtype.itemsize / 1e9:.2f} GB en {LM.dtype}")


def liberar():
    """Suelta lo que ya no se usa de la memoria (GPU y RAM)."""
    gc.collect()
    if CUDA:
        torch.cuda.empty_cache()

**La plantilla de chat.** Un modelo *Instruct* no ve tu prompt tal cual: lo ve dentro de una conversación con marcas especiales.

- `<|im_start|>system`: instrucciones generales (Qwen pone una por omisión).
- `<|im_start|>user`: lo que escribe la persona; aquí va **tu `prompt`**.
- `<|im_start|>assistant`: aquí empieza a contestar el modelo; aquí va **tu `completion`**.
- `<|im_end|>`: fin de un turno. Cuando el modelo lo genera, deja de hablar.

- `formatear(prompt)` arma ese texto con `apply_chat_template`, la plantilla que trae el propio tokenizer.
- Se usa **la misma** al entrenar y al generar. Si cambiaras la plantilla entre una cosa y otra, el modelo vería un formato que nunca practicó.

In [ ]:
def formatear(prompt):
    """El prompt dentro de la plantilla de chat del modelo, listo para que
    el asistente conteste."""
    return TOK.apply_chat_template([{"role": "user", "content": prompt}],
                                   tokenize=False, add_generation_prompt=True)


print(formatear(PROMPTS[0]))

- `generar(prompts, lora)` contesta con **greedy decoding**: en cada paso toma el token más probable. Sin azar, así que la misma entrada da siempre la misma salida y las comparaciones son justas. Qwen viene configurado para muestrear al azar (*temperature*, *top-p*); la celda de carga lo apaga.
- `max_new` (por omisión `P["max_new"]`) acota cuántos tokens puede escribir. Si la respuesta se ve cortada a media frase, es ese tope, no el modelo.
- Genera en lotes de 16; el relleno (*padding*) va a la **izquierda** para que todas las respuestas empiecen justo después de su prompt.
- `lora=False` apaga el adaptador: con **un solo modelo en la memoria** se obtienen el base y el ajustado.

In [ ]:
def _sin_adaptador(modelo, apagar):
    """Contexto que apaga LoRA (si el modelo ya lo tiene y apagar=True)."""
    if apagar and hasattr(modelo, "disable_adapter"):
        return modelo.disable_adapter()
    return contextlib.nullcontext()


@torch.no_grad()
def generar(prompts, lora=True, max_new=None, modelo=None, lote=16):
    """Respuestas greedy, una por prompt. lora=False → el modelo base."""
    prompts = [prompts] if isinstance(prompts, str) else list(prompts)
    modelo = LM if modelo is None else modelo
    cfg = dict(max_new_tokens=max_new or P["max_new"],
               pad_token_id=TOK.pad_token_id)
    TOK.padding_side = "left"            # al generar, el relleno va a la izquierda
    salida = []
    with _sin_adaptador(modelo, not lora):
        modelo.eval()
        for i in range(0, len(prompts), lote):
            x = TOK([formatear(p) for p in prompts[i:i + lote]],
                    return_tensors="pt", padding=True,
                    add_special_tokens=False).to(DEVICE)
            y = modelo.generate(**x, **cfg)
            salida += TOK.batch_decode(y[:, x["input_ids"].shape[1]:],
                                       skip_special_tokens=True)
    TOK.padding_side = "right"           # al entrenar, a la derecha
    return [s.strip() for s in salida]


def respuestas(prompts, **columnas):
    """Tabla prompt → una columna por modelo (o por lo que pases)."""
    df = pd.DataFrame({"prompt": [recorta(p, 200) for p in prompts]})
    for nombre, textos in columnas.items():
        df[nombre] = [recorta(t, 400) for t in textos]
    return df

In [ ]:
# El modelo base con tus prompts de prueba (los que NO están en el dataset).
t0 = time.perf_counter()
BASE_PROMPTS = generar(PROMPTS, lora=False)
mostrar(respuestas(PROMPTS, base=BASE_PROMPTS))

# Y tres ejemplos de validation, con la respuesta que pide tu dataset.
VAL3 = DATOS["validation"][:3]
mostrar(respuestas([e["prompt"] for e in VAL3],
                   esperado=[e["completion"] for e in VAL3],
                   base=generar([e["prompt"] for e in VAL3], lora=False)))
print(f"({time.perf_counter() - t0:.0f} s generando)")

**Cómo leer las tablas**

- Con el demo SQL, el base suele **saber** SQL, pero contesta a su manera: con un bloque de código, una explicación, a veces otra sintaxis. Lo correcto no es lo mismo que lo que tu dataset pide.
- Compara la columna `base` con `esperado`: la diferencia de **formato** es lo primero que aprende un fine-tuning.
- Con `RAPIDO=1` el modelo es diminuto y aleatorio: sus respuestas no significan nada.

## 04 · LoRA por dentro

- **Qué vas a ver:** la fórmula de LoRA y, medido en **tu** modelo, cuántos parámetros entrenas y cuánto pesa lo que te llevas.
- Las cuentas de esta sección son exactas; los números los imprime el código.

**LoRA (Low-Rank Adaptation).** Cada capa lineal elegida cambia así:

$$W=W_0+\Delta W,\qquad \Delta W=\frac{\alpha}{r}\,BA,\qquad B\in\mathbb R^{d\times r},\; A\in\mathbb R^{r\times k},\; r\ll\min(d,k).$$

- $W_0\in\mathbb R^{d\times k}$ queda **congelada**; solo se entrenan $A$ y $B$. Aquí $r=8$ y $\alpha=16$, así que $\frac{\alpha}{r}=2$.
- **Forward sin formar $\Delta W$:** $h=W_0x+\frac{\alpha}{r}\,B(Ax)$. Primero $Ax\in\mathbb R^{r}$, luego $B(Ax)\in\mathbb R^{d}$.
- **Inicio:** $A$ al azar y $B=0$ $\Rightarrow$ $\Delta W=0$. El modelo arranca **idéntico** al base; la última celda lo comprueba.
- **Rango:** $\operatorname{rank}(\Delta W)\le r$, porque todo pasa por un espacio de dimensión $r$. En §06 lo ves en los valores singulares.
- **Parámetros:** $dk$ si entrenaras $W$ completa, contra $r(d+k)$ con LoRA.

**Dónde va LoRA en un transformer.** Cada bloque tiene atención y un MLP, y los dos están hechos de lineales:

| Lineal | Dónde | Qué hace |
|---|---|---|
| `q_proj`, `k_proj`, `v_proj` | atención | arman las *queries*, *keys* y *values* |
| `o_proj` | atención | mezcla las cabezas de vuelta al ancho del modelo |
| `gate_proj`, `up_proj`, `down_proj` | MLP | el MLP de cada bloque (solo en `MODO = "largo"`) |

En Qwen2.5, `k_proj` y `v_proj` son **más angostas** que `q_proj`: varias cabezas de *query* comparten una misma *key* y *value* (*grouped-query attention*). Por eso su $d$ sale más chico en la tabla.

In [ ]:
import tempfile

from peft import LoraConfig, PeftModel, get_peft_model
from peft.tuners.lora import LoraLayer

CONFIG_LORA = LoraConfig(r=P["rango"], lora_alpha=P["alfa"],
                         target_modules=P["modulos"], lora_dropout=0.05,
                         bias="none", task_type="CAUSAL_LM")


def lora_nuevo():
    """Quita el adaptador que hubiera y pone uno nuevo: A al azar, B = 0."""
    global LM
    if isinstance(LM, PeftModel):
        LM = LM.unload()                 # el modelo base, sin capas LoRA
    semilla(0)
    LM = get_peft_model(LM, CONFIG_LORA)
    for p in LM.parameters():            # adaptador en fp32 (ver §05)
        if p.requires_grad:
            p.data = p.data.float()
    return LM


def capas_lora():
    """{nombre: capa} de cada lineal que lleva LoRA."""
    return {n: m for n, m in LM.named_modules() if isinstance(m, LoraLayer)}


lora_nuevo()
capas = capas_lora()
print(f"{len(capas)} lineales con LoRA, por ejemplo:")
for n in list(capas)[:len(P["modulos"])]:
    print("  ", n)

In [ ]:
# ¿Cuánto entrenas de verdad? Parámetros, % y MB, medidos en el modelo.
total = sum(p.numel() for p in LM.parameters())
entrenables = sum(p.numel() for p in LM.parameters() if p.requires_grad)
mb_modelo = sum(p.numel() * p.element_size() for p in LM.parameters()
                if not p.requires_grad) / 1e6
with tempfile.TemporaryDirectory() as d:  # el adaptador, guardado en disco
    LM.save_pretrained(d)
    mb_adaptador = sum(os.path.getsize(os.path.join(d, f))
                       for f in os.listdir(d)) / 1e6
display(pd.DataFrame({"valor": {
    "parámetros del modelo": f"{total - entrenables:,}",
    "parámetros entrenables (A y B)": f"{entrenables:,}",
    "% entrenable": f"{100 * entrenables / total:.3f} %",
    "MB del modelo (congelado)": f"{mb_modelo:,.0f}",
    "MB del adaptador en disco": f"{mb_adaptador:,.1f}"}}))

# Por tipo de capa, en la primera capa del modelo: d·k contra r(d + k).
filas = []
for n, m in capas.items():
    if ".layers.0." not in n:
        continue
    d, k = m.base_layer.weight.shape
    filas.append({"capa": n.split(".")[-1], "d (salida)": d, "k (entrada)": k,
                  "d·k": d * k, "r(d+k)": P["rango"] * (d + k),
                  "%": round(100 * P["rango"] * (d + k) / (d * k), 2)})
display(pd.DataFrame(filas).set_index("capa"))

In [ ]:
# Con B = 0, ΔW = 0: el modelo con LoRA da EXACTAMENTE lo mismo que el base.
x = TOK([formatear(PROMPTS[0])], return_tensors="pt",
        add_special_tokens=False).to(DEVICE)
with torch.no_grad():
    con = LM(**x).logits
    with LM.disable_adapter():
        sin = LM(**x).logits
print("max |logits con LoRA − logits base| =", (con - sin).abs().max().item())

**Cómo leer las tablas**

- `% entrenable` es la fracción del modelo que recibe gradiente. Todo lo demás está congelado y no necesita ni gradiente ni momentos de AdamW: por eso cabe en una T4.
- `MB del adaptador` es lo que descargas en §08. Compáralo con los MB del modelo: tu ajuste viaja en un archivo cientos de veces más chico, y se pone encima del modelo base de cualquiera.
- En la tabla por capa, `%` es $r(d+k)/dk$: cuanto más grande la capa, más ahorra LoRA.
- La diferencia de logits de la última celda tiene que ser **exactamente 0**: con $B=0$, la capa suma cero.

## 05 · Entrenar

- **Qué vas a ver:** un LoRA con $r=8$ que aprende tu dataset, y sus curvas de loss en train y validation.
- **La idea central: la loss solo cuenta en los tokens del completion.** El modelo lee el prompt completo, pero solo se le corrige lo que **contesta**.
- ¿Por qué? Tu dataset no le pide al modelo *escribir preguntas*, sino *responderlas*. Si la loss incluyera el prompt, gastaría capacidad en imitar tus preguntas (y en ejemplos con prompt largo, eso dominaría la loss).

Primero, cada ejemplo se vuelve tokens:

| Pieza | Qué es |
|---|---|
| `ids` | los tokens de `formatear(prompt)` seguidos de los del completion y su marca de fin de turno |
| `labels` | lo mismo, pero con **−100** en cada token del prompt |
| `cortado` | el ejemplo pasaba de `max_len` y se cortó al final |

- **−100** es la marca que `cross_entropy` ignora (`ignore_index=-100`): esos tokens no suman a la loss ni dan gradiente.
- El completion termina con `<|im_end|>`, y ese token **sí** cuenta: así el modelo aprende **cuándo callarse**.
- La tabla dice cuántos ejemplos se cortaron y cuántos se descartaron (el prompt solo ya llenaba `max_len`, así que no quedaba nada que aprender).

In [ ]:
import html

import torch.nn.functional as F
from transformers import get_cosine_schedule_with_warmup


def tokenizar(ej):
    """ids del prompt + completion y labels con -100 en el prompt."""
    p_txt = formatear(ej["prompt"])
    completo = TOK.apply_chat_template(
        [{"role": "user", "content": ej["prompt"]},
         {"role": "assistant", "content": ej["completion"]}], tokenize=False)
    if completo.startswith(p_txt):       # «completion<|im_end|>»
        c_txt = completo[len(p_txt):].rstrip("\n")
    else:                                # plantilla rara: a mano
        c_txt = ej["completion"] + TOK.eos_token
    p = TOK(p_txt, add_special_tokens=False)["input_ids"]
    c = TOK(c_txt, add_special_tokens=False)["input_ids"]
    return {"ids": (p + c)[:P["max_len"]],
            "labels": ([-100] * len(p) + c)[:P["max_len"]],
            "n_prompt": len(p), "cortado": len(p) + len(c) > P["max_len"]}


TOKS, resumen = {}, {}
for parte, ejs in DATOS.items():
    todos = [tokenizar(e) for e in ejs]
    # sin ningún token de completion (el prompt solo ya llena max_len): fuera
    TOKS[parte] = [t for t in todos if any(l != -100 for l in t["labels"])]
    resumen[parte] = {"ejemplos": len(todos),
                      "cortados": sum(t["cortado"] for t in todos),
                      "descartados": len(todos) - len(TOKS[parte]),
                      "tokens (mediana)": int(np.median(
                          [len(t["ids"]) for t in todos]))}
display(pd.DataFrame(resumen).T)
assert TOKS["train"], "ningún ejemplo de train cabe en max_len: acorta prompts"

**La máscara, a la vista.** La celda pinta un ejemplo de train token por token:

- **gris:** prompt y marcas de la plantilla; label −100; no cuentan.
- **verde:** completion y el `<|im_end|>` final; sí cuentan.

La tabla de abajo muestra la frontera: los últimos tokens del prompt (label −100) y los primeros del completion (label = su propio id).

**El corrimiento.** Un modelo causal, en la posición $i$, predice el token $i+1$. Por eso la loss compara los logits de las posiciones $0\ldots n-2$ con los labels de $1\ldots n-1$:

$$\mathcal L=\frac{1}{|C|}\sum_{i\in C}-\log p_\theta\big(x_{i}\mid x_{<i}\big),$$

con $C$ el conjunto de posiciones del completion (todas las de label $\ne -100$) en el batch. Es la **media por token**, no por ejemplo: una respuesta larga pesa más que una corta.

In [ ]:
def ver_mascara(t, antes=40, despues=60):
    """El ejemplo token por token: gris = prompt (label -100, no cuenta en la
    loss), verde = completion (sí cuenta)."""
    i0 = max(0, t["n_prompt"] - antes)
    piezas = [] if i0 == 0 else ["… "]
    for i, l in zip(t["ids"][i0:t["n_prompt"] + despues],
                    t["labels"][i0:t["n_prompt"] + despues]):
        txt = html.escape(TOK.decode([i])).replace("\n", "↵")
        fondo = "#dddddd" if l == -100 else "#a8e6b4"
        piezas.append(f'<span style="background:{fondo};color:#000;'
                      f'border:1px solid #fff;padding:0 1px">{txt}</span>')
    try:
        from IPython.display import HTML
        display(HTML("<div style='font-family:monospace;line-height:1.9'>"
                     + "".join(piezas) + "</div>"))
    except ImportError:
        pass
    n = t["n_prompt"]
    filas = [{"posición": j, "token": repr(TOK.decode([t["ids"][j]])),
              "id": t["ids"][j], "label": t["labels"][j],
              "¿cuenta en la loss?": "sí" if t["labels"][j] != -100 else "no"}
             for j in range(max(0, n - 4), min(len(t["ids"]), n + 4))]
    display(pd.DataFrame(filas).set_index("posición"))
    print(f"{len(t['ids'])} tokens: {n} de prompt (no cuentan) y "
          f"{len(t['ids']) - n} de completion (sí cuentan)")


ver_mascara(TOKS["train"][0])

In [ ]:
def lote(items):
    """Relleno a la derecha: pad en ids, 0 en la máscara, -100 en labels."""
    largo = max(len(t["ids"]) for t in items)
    ids = torch.full((len(items), largo), TOK.pad_token_id)
    labels = torch.full((len(items), largo), -100)
    mascara = torch.zeros((len(items), largo), dtype=torch.long)
    for i, t in enumerate(items):
        n = len(t["ids"])
        ids[i, :n], labels[i, :n] = torch.tensor(t["ids"]), torch.tensor(t["labels"])
        mascara[i, :n] = 1
    return {"ids": ids.to(DEVICE), "mascara": mascara.to(DEVICE),
            "labels": labels.to(DEVICE)}


def perdida_suma(b):
    """(suma de −log p del token correcto, número de tokens que cuentan).
    El logit de la posición i predice el token i + 1: de ahí el corrimiento."""
    logits = LM(input_ids=b["ids"], attention_mask=b["mascara"]).logits
    objetivo = b["labels"][:, 1:]
    suma = F.cross_entropy(logits[:, :-1].float().flatten(0, 1),
                           objetivo.flatten(), ignore_index=-100,
                           reduction="sum")
    return suma, (objetivo != -100).sum()


@torch.no_grad()
def perdida_val(parte="validation", lora=True):
    """Loss media por token de completion en los primeros n_val_loss."""
    items, suma, cuenta = TOKS[parte][:P["n_val_loss"]], 0.0, 0
    estaba = LM.training
    LM.eval()
    with _sin_adaptador(LM, not lora):
        for i in range(0, len(items), 2 * P["batch"]):
            with torch.autocast("cuda", dtype=torch.float16, enabled=CUDA):
                s, n = perdida_suma(lote(items[i:i + 2 * P["batch"]]))
            suma, cuenta = suma + s.item(), cuenta + n.item()
    LM.train(estaba)
    return suma / max(cuenta, 1)

Un paso de entrenamiento:

1. Toma `acum` batches de `batch` ejemplos y rellena a la derecha (padding con label −100, también ignorado).
2. Forward, loss de cada batch, y `backward`. Los gradientes se **acumulan**; la loss se divide entre los tokens de los `acum` batches, así que el resultado es la media sobre todos.
3. Recorta el gradiente a norma 1 y da **un** paso de AdamW.

| Pieza | Por qué |
|---|---|
| modelo fp16, adaptador fp32 | `GradScaler` no sabe desescalar gradientes fp16; $A$ y $B$ son pocos, así que en fp32 cuestan casi nada |
| `autocast` + `GradScaler` | fp16 rápido sin que los gradientes chicos se vuelvan cero; un paso con inf/NaN se salta y no cuenta |
| AdamW, `lr = 2e-4` | el estándar para LoRA en LLMs: diez veces más que en un fine-tuning completo, porque $\Delta W$ arranca en cero |
| warmup 5 % + cosine | el lr sube los primeros pasos y luego baja suave hasta 0 |
| gradient checkpointing | guarda menos activaciones y las recalcula en el backward: menos memoria, más tiempo |
| `lora_dropout = 0.05` | apaga al azar 5 % de la entrada de LoRA al entrenar; un poco de regularización |

`entrenar()` siempre empieza con un adaptador **nuevo**: si vuelves a correr la celda, no sigue entrenando el anterior.

In [ ]:
def _preparar():
    """Adaptador nuevo, checkpointing, AdamW, cosine con warmup, GradScaler."""
    lora_nuevo()
    liberar()
    if CUDA:
        torch.cuda.reset_peak_memory_stats()
    LM.gradient_checkpointing_enable(
        gradient_checkpointing_kwargs={"use_reentrant": False})
    LM.enable_input_require_grads()
    LM.config.use_cache = False           # el caché no sirve al entrenar
    entrenables = [p for p in LM.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(entrenables, lr=P["lr"], weight_decay=0.0)
    sched = get_cosine_schedule_with_warmup(
        opt, num_warmup_steps=max(1, round(0.05 * P["pasos"])),
        num_training_steps=P["pasos"])
    return entrenables, opt, sched, torch.amp.GradScaler("cuda", enabled=CUDA)


def _batches(semilla_=0):
    """Batches de train sin fin: cada época, un orden nuevo (y fijo)."""
    g, train = np.random.default_rng(semilla_), TOKS["train"]
    n = min(P["batch"], len(train))
    while True:
        orden = g.permutation(len(train))
        for i in range(0, len(orden) - n + 1, n):     # batches completos
            yield lote([train[j] for j in orden[i:i + n]])

In [ ]:
def entrenar():
    """Un LoRA desde cero. Devuelve el historial de loss train/validation."""
    semilla(0)
    entrenables, opt, sched, escalador = _preparar()
    hist = {"train": [], "validation": [(0, perdida_val())], "seg": []}
    fuente, paso, intentos = _batches(), 0, 0
    LM.train()
    while paso < P["pasos"] and intentos < 2 * P["pasos"]:
        intentos += 1
        t0 = time.perf_counter()
        lotes = [next(fuente) for _ in range(P["acum"])]
        n_tok = sum((b["labels"][:, 1:] != -100).sum() for b in lotes)
        opt.zero_grad(set_to_none=True)
        total = 0.0
        for b in lotes:                   # acumula acum gradientes
            with torch.autocast("cuda", dtype=torch.float16, enabled=CUDA):
                s, _ = perdida_suma(b)
            perdida = s / n_tok           # media sobre TODOS los tokens del paso
            escalador.scale(perdida).backward()
            total += perdida.item()
        escalador.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(entrenables, 1.0)
        escala_previa = escalador.get_scale()
        escalador.step(opt)
        escalador.update()
        if escalador.get_scale() < escala_previa:   # inf/NaN: paso saltado
            continue
        paso += 1
        sched.step()
        if CUDA:
            torch.cuda.synchronize()
        hist["seg"].append(time.perf_counter() - t0)
        hist["train"].append((paso, total))
        if paso % P["val_cada"] == 0 or paso == P["pasos"]:
            hist["validation"].append((paso, perdida_val()))
    if paso < P["pasos"]:
        print(f"¡Ojo! solo {paso} pasos válidos de {P['pasos']} (inf/NaN)")
    LM.eval()
    LM.config.use_cache = True            # para generar rápido
    return hist

In [ ]:
inicio = time.perf_counter()
HIST = entrenar()
seg = HIST["seg"][3:] or HIST["seg"]
print(f"✓ {len(HIST['train'])} pasos en {time.perf_counter() - inicio:.0f} s "
      f"({np.mean(seg):.2f} s/paso)")
print(f"loss de validation: {HIST['validation'][0][1]:.3f} (base) → "
      f"{HIST['validation'][-1][1]:.3f} (LoRA)")
if CUDA:
    print(f"memoria pico de la GPU: {torch.cuda.max_memory_allocated() / 1e9:.1f} GB")

In [ ]:
def curvas(hist, titulo="Loss solo en el completion"):
    """Train delgada y tenue (un batch por punto); validation gruesa."""
    fig, eje = plt.subplots(figsize=(7.5, 3.6))
    eje.plot(*zip(*hist["train"]), color=COLORES["train"], lw=0.8,
             alpha=0.5, label="train (cada paso)")
    eje.plot(*zip(*hist["validation"]), color=COLORES["validation"], lw=2.5,
             marker="o", ms=4, label="validation")
    eje.set(xlabel="paso", ylabel="loss por token (nats)", title=titulo)
    eje.legend()
    plt.tight_layout()
    plt.show()


curvas(HIST)

**Cómo leer la gráfica**

- **Validation** (naranja, gruesa) es la que importa. Su primer punto, en el paso 0, es el **modelo base**: todo lo que baja de ahí es lo que aprendió.
- **Train** (azul, delgada) brinca porque cada punto es un solo batch. Mira su tendencia, no cada punto.
- La unidad es *nats* por token: $-\log p$ promedio. Una loss de $0.7$ quiere decir que el token correcto recibe, en promedio geométrico, probabilidad $e^{-0.7}\approx 0.5$.
- **Si validation sube mientras train baja**, el modelo empezó a memorizar train: menos `pasos` o más datos (ver la receta de §08).
- **Si casi no baja**, puede que el base ya contestara así, o que la tarea sea muy difícil para un modelo de 0.5B. No siempre sale un efecto grande, y eso también es un resultado.

## 06 · Ver LoRA por dentro

- **Qué vas a ver:** cuatro cosas que solo tiene LoRA, medidas en el adaptador que acabas de entrenar.

| | Qué | Qué esperar |
|---|---|---|
| (a) | valores singulares de $\Delta W$ | caen a cero después del $r$-ésimo |
| (b) | $\lVert\Delta W\rVert_F/\lVert W_0\rVert_F$ por capa | cambios **chicos**; unas capas más que otras |
| (c) | la perilla $\lambda$ | $\lambda=0$ es el base; $\lambda$ grande exagera el ajuste |
| (d) | `merge_and_unload` | misma salida que el adaptador, sin capas extra |

La primera celda calcula, para cada capa con LoRA, $\Delta W=\frac{\alpha}{r}BA$ y su tamaño relativo a $W_0$.

In [ ]:
def delta_w(capa):
    """ΔW = (α/r)·B·A de una capa LoRA, en CPU y fp32: [d_salida, d_entrada]."""
    A = capa.lora_A["default"].weight.detach().float().cpu()
    B = capa.lora_B["default"].weight.detach().float().cpu()
    return capa.scaling["default"] * (B @ A)


filas = []
for n, m in capas_lora().items():
    dw, w0 = delta_w(m), m.base_layer.weight.detach().float().cpu()
    filas.append({"nombre": n, "bloque": int(n.split(".layers.")[1].split(".")[0]),
                  "lineal": n.split(".")[-1],
                  "relativa": (dw.norm() / w0.norm()).item()})
NORMAS = pd.DataFrame(filas)
mayor = NORMAS.loc[NORMAS["relativa"].idxmax()]
print(f"La capa que más cambió: {mayor['nombre']} "
      f"(‖ΔW‖/‖W₀‖ = {mayor['relativa']:.4f})")

**(a) El rango de $\Delta W$.** Una matriz $d\times k$ cualquiera tiene hasta $\min(d,k)$ valores singulares distintos de cero. $\Delta W=\frac{\alpha}{r}BA$ pasa por un cuello de ancho $r$, así que **a lo más $r$**.

- La gráfica divide cada $\sigma_i$ entre el mayor, $\sigma_1$, y usa escala logarítmica.
- **Gris, $W_0$:** decae poco a poco; la matriz original usa muchas direcciones.
- **Naranja, $\Delta W$:** $r$ valores y luego un **precipicio** hasta $\approx 10^{-7}$, el cero de la aritmética fp32.

**Cómo leer la gráfica:** todo lo que aprendió esta capa vive en $r=8$ direcciones. Eso es lo «low-rank» de LoRA.

In [ ]:
# (a) Valores singulares de ΔW en la capa que más cambió, y de su W₀.
capa = capas_lora()[mayor["nombre"]]
s_dw = torch.linalg.svdvals(delta_w(capa))
s_w0 = torch.linalg.svdvals(capa.base_layer.weight.detach().float().cpu())
r, k = P["rango"], min(3 * P["rango"], len(s_dw))
fig, eje = plt.subplots(figsize=(7.5, 3.4))
eje.semilogy(range(1, k + 1), s_w0[:k] / s_w0[0], "o-", color="gray",
             ms=3, label="W₀ (congelada)")
eje.semilogy(range(1, k + 1), (s_dw[:k] / s_dw[0]).clamp_min(1e-12), "o-",
             color=COLORES["validation"], ms=4, label="ΔW (LoRA)")
eje.axvline(r + 0.5, color="black", ls="--", lw=1)
eje.text(r + 0.7, 1e-3, f"r = {r}", fontsize=9)
eje.set(xlabel="índice del valor singular", ylabel="σᵢ / σ₁ (escala log)",
        title=f"Valores singulares · {mayor['lineal']} del bloque "
              f"{mayor['bloque']}")
eje.legend()
plt.tight_layout()
plt.show()
if len(s_dw) > r:
    print(f"σ_{r + 1} / σ_1 de ΔW = {(s_dw[r] / s_dw[0]).item():.1e}  "
          f"(cero numérico: rank(ΔW) ≤ {r})")

**(b) Qué capas cambiaron más.** Para cada lineal con LoRA, $\lVert\Delta W\rVert_F/\lVert W_0\rVert_F$: cuánto se movió respecto a su propio tamaño.

**Cómo leer la gráfica**

- Una barra por lineal en cada bloque; el eje x va de la entrada (bloque 0) a la salida del modelo.
- Los valores suelen ser **chicos** (del orden de centésimas o menos): LoRA retoca, no reescribe.
- Que unas lineales cambien más que otras depende de tu tarea y de la escala de cada $W_0$. No hay un patrón que *deba* salir; compara la tabla de promedios entre lineales.

In [ ]:
# (b) ‖ΔW‖_F / ‖W₀‖_F por bloque y por lineal.
tabla_n = NORMAS.pivot(index="bloque", columns="lineal", values="relativa")
tabla_n = tabla_n[[m for m in P["modulos"] if m in tabla_n.columns]]
ancho = 0.8 / len(tabla_n.columns)
fig, eje = plt.subplots(figsize=(10, 3.6))
for j, lineal in enumerate(tabla_n.columns):
    eje.bar(tabla_n.index + (j - (len(tabla_n.columns) - 1) / 2) * ancho,
            tabla_n[lineal], width=ancho, color=plt.cm.tab10(j), label=lineal)
eje.set(xlabel="bloque del transformer (0 = el más cercano a la entrada)",
        ylabel="‖ΔW‖ / ‖W₀‖", title="¿Qué capas cambió más tu LoRA?")
eje.legend(ncol=len(tabla_n.columns), fontsize=8)
plt.tight_layout()
plt.show()
display(tabla_n.mean().rename("promedio por lineal").to_frame().T.round(4))

**(c) La perilla $\lambda$.** Al generar, cada capa calcula

$$h=W_0x+\lambda\,\frac{\alpha}{r}\,B(Ax).$$

- $\lambda=0$: el modelo **base** (la celda lo comprueba contra `lora=False`).
- $\lambda=1$: el adaptador tal como se entrenó.
- $\lambda=0.5$: medio ajuste. $\lambda=1.5$: el ajuste **exagerado**.
- `escala_lora(λ)` multiplica el `scaling` que peft guarda en cada capa (el $\frac{\alpha}{r}$) y lo restaura al salir.

**Cómo leer la tabla:** cada columna es la misma entrada con otro $\lambda$. Busca en qué $\lambda$ aparece el formato de tu dataset. Con $\lambda=1.5$ a veces sale igual que con 1 y a veces se descompone (repite, o se corta): el adaptador se entrenó para $\lambda=1$.

In [ ]:
@contextlib.contextmanager
def escala_lora(lam):
    """Multiplica α/r de TODAS las capas LoRA por λ; al salir, lo restaura."""
    capas = list(capas_lora().values())
    antes = [dict(m.scaling) for m in capas]
    try:
        for m in capas:
            for nombre in m.scaling:
                m.scaling[nombre] *= lam
        yield
    finally:
        for m, a in zip(capas, antes):
            m.scaling.update(a)


# (c) La perilla: el mismo prompt, greedy (sin azar), cuatro valores de λ.
PROMPTS_L = [PROMPTS[0], DATOS["validation"][0]["prompt"]]
por_lambda = {}
for lam in [0, 0.5, 1, 1.5]:
    with escala_lora(lam):
        por_lambda[f"λ = {lam}"] = generar(PROMPTS_L)
mostrar(respuestas(PROMPTS_L, **por_lambda))
print("λ = 0 da lo mismo que el base:",
      por_lambda["λ = 0"] == generar(PROMPTS_L, lora=False))

**(d) Fusionar.** `merge_and_unload()` calcula $W=W_0+\frac{\alpha}{r}BA$ una vez y deja un modelo **normal**, sin capas LoRA: misma forma y mismos parámetros que el base, y ningún costo extra al generar.

- Se hace sobre una **copia**: fusionar destruye el adaptador, y lo necesitas en §07 y §08.
- Los logits no salen idénticos al bit: $W_0$ está en fp16, y al sumarle $\Delta W$ el resultado se **redondea** a fp16. Compara la diferencia con la escala de los logits: es ruido de redondeo.
- Las respuestas greedy casi siempre coinciden. Si alguna difiere, fue un empate casi perfecto entre dos tokens que el redondeo rompió hacia el otro lado.

In [ ]:
# (d) merge_and_unload sobre una COPIA: W = W₀ + (α/r)BA en una sola matriz.
import copy

copia = copy.deepcopy(LM)
fusionado = copia.merge_and_unload()
print("capas LoRA que quedan tras fusionar:",
      sum(isinstance(m, LoraLayer) for m in fusionado.modules()))
print(f"parámetros: base {n_lm:,} · fusionado "
      f"{sum(p.numel() for p in fusionado.parameters()):,}")
TOK.padding_side = "left"
x = TOK([formatear(p) for p in PROMPTS], return_tensors="pt", padding=True,
        add_special_tokens=False).to(DEVICE)
TOK.padding_side = "right"
with torch.no_grad():
    con_adaptador = LM(**x).logits.float()
    con_fusion = fusionado(**x).logits.float()
dif = (con_adaptador - con_fusion).abs().max().item()
print(f"max |logits adaptador − logits fusionado| = {dif:.2e} "
      f"(escala de los logits: {con_adaptador.abs().max().item():.1f})")
iguales = generar(PROMPTS) == generar(PROMPTS, modelo=fusionado)
print("¿mismas respuestas greedy en prompts.txt?", iguales)
del copia, fusionado, con_adaptador, con_fusion
liberar()

## 07 · Métricas, explicadas

- **Qué vas a ver:** cuatro números para comparar el base contra tu LoRA, cada uno con lo que mide y **cómo te puede engañar**.
- Todo se mide en **validation**. El test espera a §08.
- Un solo número nunca basta: la loss puede bajar sin que las respuestas mejoren, y una respuesta correcta puede tener exact match 0. Por eso van juntos, y al final de la sección hay una tabla de ejemplos para **leer**.

**Loss y perplexity (solo en el completion)**

$$\mathcal L=\frac{1}{|C|}\sum_{i\in C}-\log p_\theta(x_i\mid x_{<i}),\qquad \text{perplexity}=e^{\mathcal L}.$$

| Qué mide | Rango | Qué es bueno | Trampa |
|---|---|---|---|
| qué tan probable le parece al modelo **la respuesta de tu dataset**, token por token | loss $\ge 0$; perplexity $\ge 1$ | más bajo | mide si el modelo *esperaba* esa respuesta, no si la que **genera** es buena; no se compara entre tokenizers distintos ni entre datasets |

- La perplexity se lee como «entre cuántos tokens duda, en promedio». Perplexity 2 ≈ dudar entre dos opciones en cada token.
- Es la única métrica que no necesita generar: es rápida y suave, por eso es la de las curvas.

**Exact match (normalizado)**

| Qué mide | Rango | Qué es bueno | Trampa |
|---|---|---|---|
| la respuesta generada es **idéntica** a la esperada, tras pasar a minúsculas y colapsar espacios | 0 a 100 % | más alto | todo o nada: `age >= 20` contra `age > 20` vale 0, igual que una respuesta absurda; y una respuesta **correcta pero escrita distinto** (otro orden, un alias) también vale 0 |

- Tiene sentido cuando las respuestas son **cortas y tienen una forma canónica**, como el SQL del demo.
- Con poemas, resúmenes o explicaciones va a dar ≈ 0 para cualquier modelo: no la uses ahí.

**Token F1 (estilo SQuAD)**

Con $p$ las palabras de la respuesta generada y $r$ las de la esperada (como bolsas, contando repeticiones):

$$\text{precision}=\frac{|p\cap r|}{|p|},\qquad \text{recall}=\frac{|p\cap r|}{|r|},\qquad F_1=\frac{2\cdot\text{precision}\cdot\text{recall}}{\text{precision}+\text{recall}}.$$

| Qué mide | Rango | Qué es bueno | Trampa |
|---|---|---|---|
| cuántas **palabras** comparten la respuesta generada y la esperada: crédito parcial | 0 a 100 % | más alto | ignora el orden: `SELECT a FROM b` y `FROM a SELECT b` tienen F1 = 100 %; y premia copiar palabras del prompt aunque la respuesta esté mal |

- «Palabras» aquí son los pedazos separados por espacios, tras normalizar. A diferencia de SQuAD, no se quita la puntuación ni los artículos: en SQL, `>` y `>=` no son lo mismo.
- La celda calcula las dos métricas en un ejemplo a mano: una diferencia mínima da exact match 0 y F1 alto.

El **olvido** (¿perdió lo que ya sabía?) no tiene número: se lee en §08, comparando respuestas a preguntas generales.

In [ ]:
from collections import Counter


def normalizar(texto):
    """Minúsculas y espacios colapsados (ni puntuación ni artículos se tocan)."""
    return " ".join(texto.lower().split())


def exact_match(pred, real):
    return float(normalizar(pred) == normalizar(real))


def token_f1(pred, real):
    """F1 entre las bolsas de palabras de pred y real (estilo SQuAD)."""
    p, r = normalizar(pred).split(), normalizar(real).split()
    if not p or not r:
        return float(p == r)
    comunes = sum((Counter(p) & Counter(r)).values())
    if comunes == 0:
        return 0.0
    precision, recall = comunes / len(p), comunes / len(r)
    return 2 * precision * recall / (precision + recall)


# Un ejemplo a mano, para ver la diferencia entre las dos:
real = "SELECT name FROM students WHERE age > 20"
pred = "select name from students where age >= 20"
print(f"exact match = {exact_match(pred, real):.0f} · "
      f"token F1 = {token_f1(pred, real):.3f}")

In [ ]:
def evaluar(parte):
    """Base y LoRA en los primeros n_eval ejemplos de `parte`: una tabla con
    las cuatro métricas y las respuestas de cada modelo."""
    ejs = DATOS[parte][:P["n_eval"]]
    reales = [e["completion"] for e in ejs]
    filas, preds = {}, {}
    for nombre, lora in [("base", False), ("LoRA", True)]:
        t0 = time.perf_counter()
        loss = perdida_val(parte, lora=lora)
        preds[nombre] = generar([e["prompt"] for e in ejs], lora=lora)
        filas[nombre] = {
            "loss": loss, "perplexity": math.exp(min(loss, 50)),
            "exact match %": 100 * np.mean(
                [exact_match(p, r) for p, r in zip(preds[nombre], reales)]),
            "token F1 %": 100 * np.mean(
                [token_f1(p, r) for p, r in zip(preds[nombre], reales)])}
        print(f"{nombre}: {time.perf_counter() - t0:.0f} s")
    print(f"({len(ejs)} ejemplos de {parte} para generar, "
          f"{min(len(TOKS[parte]), P['n_val_loss'])} para la loss)")
    return pd.DataFrame(filas).T, preds


def por_ejemplo(parte, preds, n=5):
    """Las primeras n respuestas, con su exact match y su F1."""
    ejs = DATOS[parte][:min(n, len(preds["base"]))]
    df = respuestas([e["prompt"] for e in ejs],
                    esperado=[e["completion"] for e in ejs],
                    base=preds["base"][:n], LoRA=preds["LoRA"][:n])
    df["EM base / LoRA"] = [
        f"{exact_match(b, e['completion']):.0f} / "
        f"{exact_match(l, e['completion']):.0f}"
        for b, l, e in zip(preds["base"], preds["LoRA"], ejs)]
    df["F1 base / LoRA"] = [
        f"{token_f1(b, e['completion']):.2f} / "
        f"{token_f1(l, e['completion']):.2f}"
        for b, l, e in zip(preds["base"], preds["LoRA"], ejs)]
    return df

In [ ]:
VAL_TABLA, VAL_PRED = evaluar("validation")
display(VAL_TABLA.round(3))
mostrar(por_ejemplo("validation", VAL_PRED))

**Cómo leer la tabla**

- Compara cada columna fila contra fila: **base** contra **LoRA**. En la loss y la perplexity, LoRA debe salir más baja; en exact match y F1, más alta.
- Si la loss bajó mucho pero el exact match no se movió, lee los ejemplos: a veces el modelo ya contesta casi bien y falla en un detalle (un espacio, una comilla). Para eso está F1.
- **Si el base ya tenía un F1 alto**, el ajuste tenía poco que ganar en esa métrica; mira la loss y el formato.
- Estos números son de **validation**, que usaste para mirar curvas y la perilla. El número honesto es el de test, en §08.

## 08 · Testeo, una sola vez

- **Esta es la única vez que se usa el test.** Todo lo anterior (curvas, métricas, la perilla) se decidió con validation. Ya no cambies nada por lo que veas aquí: si lo haces, este número deja de ser honesto.
- La tabla tiene las mismas métricas de §07, ahora en ejemplos que **nadie** miró antes.
- Esperamos números parecidos a los de validation. Si test sale mucho peor, validation se usó de más para decidir (o los dos conjuntos son muy chicos y el azar pesa).

In [ ]:
# El test, por primera y única vez.
TEST_TABLA, TEST_PRED = evaluar("test")
display(TEST_TABLA.round(3))
mostrar(por_ejemplo("test", TEST_PRED))

**Tus prompts, antes y después.** Los de `prompts.txt` no están en el dataset: es lo más cercano a cómo usarías el modelo de verdad. La columna `base` es la misma de §03.

In [ ]:
# Tus prompts de prueba: el base de §03 contra tu LoRA.
mostrar(respuestas(PROMPTS, base=BASE_PROMPTS, LoRA=generar(PROMPTS)))

**Olvido.** Ajustar un modelo a tu tarea puede hacerle **perder** algo de lo que ya sabía (*catastrophic forgetting*). LoRA lo limita, porque $W_0$ no se mueve, pero no lo impide.

| Qué mide | Rango | Qué es bueno | Trampa |
|---|---|---|---|
| si el modelo sigue contestando bien preguntas **ajenas** a tu dataset | sin número: se lee | respuestas de LoRA tan buenas como las del base | tres preguntas son una muestra mínima; y un LoRA que contesta todo **en el formato de tu dataset** (todo en SQL, todo en verso) también es olvido, aunque la respuesta no sea falsa |

**Cómo leer la tabla:** busca si LoRA cambió el estilo (más corto, en el formato de tus datos) o el contenido (datos falsos). Lo primero es común y a veces es lo que querías; lo segundo es olvido de verdad.

In [ ]:
# Tres preguntas que no tienen nada que ver con tu dataset.
AJENAS = ["¿Cuál es la capital de Francia? Contesta en una frase.",
          "Explica en dos frases qué es la fotosíntesis.",
          "Escribe un haiku sobre el mar."]
mostrar(respuestas(AJENAS, base=generar(AJENAS, lora=False, max_new=96),
                   LoRA=generar(AJENAS, max_new=96)))

**Tu adaptador.** Se guarda solo lo que entrenaste: $A$ y $B$ de cada capa (`adapter_model.safetensors`), la configuración (`adapter_config.json`) y una ficha con tu dataset y tus números de test. Lo descargas como `mi_lora_texto.zip`.

Para usarlo otro día, en cualquier máquina con el mismo modelo base:

```python
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer
base = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-0.5B-Instruct")
modelo = PeftModel.from_pretrained(base, "mi_lora_texto")
tok = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B-Instruct")
```

(En `MODO = "largo"`, el base es `Qwen/Qwen2.5-1.5B-Instruct`.) Si Colab no abre la descarga, búscala en el panel 📁 Archivos: clic derecho → Descargar.

In [ ]:
# Solo A y B (y su configuración) a una carpeta, y la carpeta a un zip.
import shutil

CARPETA = "mi_lora_texto"
LM.save_pretrained(CARPETA)
with open(os.path.join(CARPETA, "LEEME.md"), "w", encoding="utf-8") as f:
    f.write(f"# Mi LoRA de texto\n\n- Modelo base: {MODELO}\n"
            f"- r = {P['rango']}, α = {P['alfa']}, capas: {P['modulos']}\n"
            f"- {len(HIST['train'])} pasos, lr = {P['lr']}\n\n"
            f"## Mi dataset\n\n{LEEME}\n\n## Test (una vez)\n\n"
            f"{TEST_TABLA.round(3).to_string()}\n")
zip_lora = shutil.make_archive(CARPETA, "zip", CARPETA)
print(f"{zip_lora}: {os.path.getsize(zip_lora) / 1e6:.1f} MB")
try:
    from google.colab import files
    files.download(zip_lora)              # baja a tu computadora
except ImportError:
    print("(fuera de Colab: el zip queda en la carpeta actual)")

**Si salió mal, qué mover**

| Lo que ves | Qué pasa | Qué mover |
|---|---|---|
| validation casi no baja desde el paso 0 | el base ya contestaba así, o la tarea es difícil para este modelo | más `pasos`; `MODO = "largo"` (modelo de 1.5B y LoRA en el MLP) |
| train baja y validation sube al final | memoriza train | menos `pasos`; más datos |
| loss en `nan` o el aviso de pasos saltados | fp16 se desbordó | baja `lr` a `1e-4` |
| las respuestas no paran hasta `max_new` | no aprendió a terminar | más `pasos`; revisa que tus completions no traigan relleno al final |
| las respuestas salen cortadas a media frase | el tope `max_new` | sube `P["max_new"]` |
| muchos ejemplos «cortados» en §05 | `max_len` corta tus ejemplos | acorta prompts o completions; o sube `P["max_len"]` (más memoria) |
| exact match 0 con F1 alto | casi bien, falla un detalle | lee los ejemplos; quizá tu tarea pide F1, no exact match |
| LoRA contesta todo en tu formato (§08, olvido) | se pasó de ajuste | menos `pasos`, o $\lambda<1$ al generar (§06) |
| memoria agotada (*CUDA out of memory*) | ejemplos largos o modelo grande | baja `P["batch"]` a 2 y sube `P["acum"]` a 4: mismo batch efectivo |

**Lo que te llevas**

- Un LoRA de $r=8$ mueve una fracción mínima del modelo y aun así cambia **cómo contesta**: primero el formato, luego el contenido.
- La loss solo en el completion le enseña a responder, no a repetir tus preguntas.
- $\Delta W$ tiene rango $\le r$, se puede **subir o bajar** con $\lambda$, y se **fusiona** sin costo.
- Cada métrica tiene su trampa: loss y perplexity no generan, exact match es todo o nada, F1 ignora el orden, y el olvido hay que leerlo.
- El test se usa **una vez**. Lo que muestres de este notebook es ese número.